# Week 24, Day 5: Diffusion models: generating data by reversing noise

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w024/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Noising in one step or a hundred

Compute `abar` as the cumulative product of alpha = 1 − beta. Draw 20,000 data points with `data(n)`, then make `xT_closed` with the closed form at t = T, and `xT_steps` by applying the one-step update 100 times.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
T = 100
beta = np.linspace(1e-4, 0.1, T); alpha = 1 - beta
abar = ...
r_d = np.random.default_rng(2020)
def data(n):
    return np.where(r_d.random(n) < 0.5, -2.0, 2.0) + 0.1 * r_d.normal(size=n)
x0 = data(20000)
xT_closed = ...
x = x0.copy()
for t in range(T):
    x = ...
xT_steps = x
print(f"abar_T = {abar[-1]:.4f};  closed form: mean {xT_closed.mean():.3f}, var {xT_closed.var():.3f};  stepped: var {xT_steps.var():.3f}")

In [ ]:
import numpy as np
def test_closed():
    "the closed-form x_T has mean within 0.05 of 0 and variance within 0.05 of abar_T var(x0) + 1 - abar_T"
    target = abar[-1] * x0.var() + 1 - abar[-1]
    assert abs(xT_closed.mean()) < 0.05 and abs(xT_closed.var() - target) < 0.05

def test_same():
    "stepping 100 times gives the same variance as the closed form, within 0.05"
    assert abs(xT_steps.var() - xT_closed.var()) < 0.05

test_closed()
test_same()
print("All checks passed")

### Exercise: Sampling with the exact score

Write `eps_exact(x, t)` from the formulas above. Starting from 4,000 standard normal draws, run the reverse update from t = T − 1 down to 0 (indices into the arrays), adding sqrt(beta[t]) times fresh noise except at t = 0, and store the result in `x_exact`.

In [ ]:
def eps_exact(x, t):
    a = abar[t]
    m0 = 2 * np.tanh(2 * np.sqrt(a) * x / (1 - a))     # posterior mean of x0
    return ...

r_x = np.random.default_rng(1)
x = r_x.normal(size=4000)
for t in range(T - 1, -1, -1):
    x = ...
    if t > 0:
        x = x + np.sqrt(beta[t]) * r_x.normal(size=x.size)
x_exact = x
print(f"distinct values (rounded to 6 places): {np.unique(np.round(x_exact, 6))};  fraction positive {np.mean(x_exact > 0):.3f}")

In [ ]:
import numpy as np
def test_copies():
    "every sample lands within 1e-6 of a training point, -2 or +2"
    assert np.all(np.abs(np.abs(x_exact) - 2) < 1e-6)

def test_balance():
    "the two training points are produced in roughly equal numbers (40 to 60%)"
    assert 0.4 < np.mean(x_exact > 0) < 0.6

test_copies()
test_balance()
print("All checks passed")

### Exercise: A diffusion model by least squares

For each t, noise 4,000 fresh data points to level t and fit weights W[t] by np.linalg.lstsq of the features against the clean x0. Then generate 4,000 samples: at each step predict x0, clip it to [−3, 3], convert it to a noise prediction, apply the reverse update, and store the result in `x_gen`.

In [ ]:
r_l = np.random.default_rng(2006)
def data_l(n):
    return np.where(r_l.random(n) < 0.5, -2.0, 2.0) + 0.1 * r_l.normal(size=n)
scales = np.array([0.25, 0.5, 1, 2, 4, 8])
feat = lambda x: np.column_stack([np.ones_like(x), x] + [np.tanh(s * x) for s in scales])
W = []
for t in range(T):
    x0_t = data_l(4000); e = r_l.normal(size=4000)
    xt = np.sqrt(abar[t]) * x0_t + np.sqrt(1 - abar[t]) * e
    W.append(np.linalg.lstsq(feat(xt), x0_t, rcond=None)[0])

x = r_l.normal(size=4000)
for t in range(T - 1, -1, -1):
    x0_hat = np.clip(feat(x) @ W[t], -3, 3)
    eps_hat = ...                                 # the noise implied by x and x0_hat
    x = (x - beta[t] / np.sqrt(1 - abar[t]) * eps_hat) / np.sqrt(alpha[t])
    if t > 0:
        x = x + np.sqrt(beta[t]) * r_l.normal(size=x.size)
x_gen = x
print(f"near a cluster: {np.mean(np.abs(x_gen) > 1):.3f};  positive: {np.mean(x_gen > 0):.3f};  mean |x| {np.abs(x_gen).mean():.3f}, sd of |x| {np.std(np.abs(x_gen)):.3f}")
plt.hist(data_l(4000), 80, alpha=0.5, density=True, label="data"); plt.hist(x_gen, 80, alpha=0.5, density=True, label="generated")
plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_modes():
    "over 95% of generated samples are near a cluster (|x| above 1), split between them 40 to 60%"
    assert np.mean(np.abs(x_gen) > 1) > 0.95 and 0.4 < np.mean(x_gen > 0) < 0.6

def test_spread():
    "the samples have spread about each cluster, with sd of |x| between 0.05 and 0.2, like the data's 0.1"
    assert 0.05 < np.std(np.abs(x_gen)) < 0.2

test_modes()
test_spread()
print("All checks passed")